# CRNN vs YOLO — Kaggle Starter Pipeline

Step-by-step starter notebook for [CraftyCode121/crnn-vs-yolo](https://github.com/CraftyCode121/crnn-vs-yolo). Each stage is split into its own Markdown and code cells so you can run, inspect, and rerun individual steps.

**Kaggle setup:** set Accelerator to **GPU** for training and enable **Internet** if you need to clone the repository or install packages. Long-running stages are disabled by default.

> This notebook avoids guessing project-specific CLI flags for dataset generation or evaluation. It inspects the current checkout first; confirm each command against the repository version you cloned before running a long job.

## 1. Runtime switches and paths

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, textwrap, time

REPO_URL = "https://github.com/CraftyCode121/crnn-vs-yolo.git"
REPO_DIR = Path("/kaggle/working/crnn-vs-yolo")

# Set True only for stages you actually want to run.
INSTALL_REQUIREMENTS = True
GENERATE_SYNTHETIC_DATA = False
TRAIN_CRNN = False
TRAIN_YOLO = False
EVALUATE_CRNN = False
EVALUATE_YOLO = False

CRNN_CONFIG = "crnn/training.yaml"
CRNN_EPOCHS = 50  # verify config/CLI supports overriding before use
YOLO_MODEL = "yolo11s.pt"
YOLO_EPOCHS = 50
YOLO_IMAGE_SIZE = 640
YOLO_BATCH_SIZE = 16

# Optional: directory attached under /kaggle/input with prepared data.
EXTERNAL_DATA_DIR = None

print("Python:", sys.version)
print("Current directory:", Path.cwd())
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout[:2500] or "No nvidia-smi output; check Kaggle accelerator settings.")
except FileNotFoundError:
    print("nvidia-smi not found; check Kaggle accelerator settings.")

## 2. Clone the repository

In [ ]:
if REPO_DIR.exists():
    print(f"Repository already exists: {REPO_DIR}")
    print(subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"], capture_output=True, text=True).stdout)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Repository root:", Path.cwd())
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=False)

## 3. Inspect the repository layout

In [ ]:
for p in sorted(REPO_DIR.iterdir()):
    print(("[dir]  " if p.is_dir() else "[file] " ) + p.name)

important = [
    "README.md", "requirements.txt", "pyproject.toml",
    "synth_pages/config.yaml", "synth_pages/split_fonts.py",
    "crnn/training.yaml", "crnn/train.py", "crnn/evaluate.py",
    "yolo/data.yaml"
]
print("\nKey files:")
for rel in important:
    p = REPO_DIR / rel
    print(f"{'OK' if p.exists() else 'missing':7} {rel}")

## 4. Read the project instructions

In [ ]:
readme = REPO_DIR / "README.md"
if readme.exists():
    print(readme.read_text(errors="replace")[:12000])
else:
    print("README.md not found. Inspect repository files above.")

## 5. Install dependencies

In [ ]:
if INSTALL_REQUIREMENTS:
    requirements = REPO_DIR / "requirements.txt"
    if requirements.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
    else:
        print("No requirements.txt found. Installing common pipeline dependencies only.")
        subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics", "PyYAML", "Pillow", "opencv-python-headless", "tqdm"], check=True)
else:
    print("Dependency installation skipped.")

## 6. Verify the runtime

In [ ]:
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("PyTorch check failed:", repr(e))

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
    print("TensorFlow GPUs:", tf.config.list_physical_devices("GPU"))
except Exception as e:
    print("TensorFlow check failed:", repr(e))

## 7. Inspect dataset inputs and generator files

In [ ]:
if EXTERNAL_DATA_DIR:
    data_root = Path(EXTERNAL_DATA_DIR)
    if not data_root.exists(): raise FileNotFoundError(f"External data path does not exist: {data_root}")
    print("External dataset:", data_root)
    for p in list(data_root.iterdir())[:40]: print(p)
else:
    data_root = REPO_DIR / "synth_pages" / "output"
    print("Expected repository dataset path:", data_root)

for rel in ["synth_pages/config.yaml", "synth_pages/split_fonts.py", "synth_pages/output/crnn/labels.tsv", "synth_pages/output/metadata.jsonl", "synth_pages/output/yolo"]:
    p = REPO_DIR / rel
    print(f"{'OK' if p.exists() else 'missing':7} {rel}")

## 8. Review synthetic-data configuration

In [ ]:
generator_config = REPO_DIR / "synth_pages" / "config.yaml"
if generator_config.exists():
    print(generator_config.read_text(errors="replace")[:12000])
else:
    print("synth_pages/config.yaml not found in this checkout.")

generator_dir = REPO_DIR / "synth_pages"
if generator_dir.exists():
    print("\nGenerator files:")
    for p in sorted(generator_dir.iterdir()): print(p.name)

## 9. Generate synthetic pages (optional)

**Off by default.** Check the generator's README/help and configuration above before enabling this stage. This guard intentionally stops rather than guessing a command or accidentally regenerating thousands of pages.

In [ ]:
if GENERATE_SYNTHETIC_DATA:
    raise NotImplementedError(
        "Confirm the exact generator command from this checkout's synth_pages instructions, then place it here. "
        "Do not rerun full generation if a prepared dataset is already attached."
    )
else:
    print("Skipped. Set GENERATE_SYNTHETIC_DATA=True only after adding the verified command.")

## 10. Inspect CRNN training configuration and data paths

In [ ]:
crnn_config_path = REPO_DIR / CRNN_CONFIG
if crnn_config_path.exists():
    print(crnn_config_path.read_text(errors="replace")[:16000])
else:
    print(f"CRNN config not found: {crnn_config_path}")

for rel in ["synth_pages/output/crnn/labels.tsv", "synth_pages/output/metadata.jsonl", "crnn/checkpoints", "crnn/reports", "crnn/logs"]:
    p = REPO_DIR / rel
    print(f"{'OK' if p.exists() else 'missing':7} {rel}")

## 11. CRNN training

The project has been run from the repository root using `python -m crnn.train --config crnn/training.yaml`. Run the help cell first to confirm the current CLI. Then enable training only when the dataset paths and GPU are correct.

In [ ]:
train_module_exists = (REPO_DIR / "crnn" / "train.py").exists()
print("crnn/train.py exists:", train_module_exists)
if train_module_exists:
    subprocess.run([sys.executable, "-m", "crnn.train", "--help"], check=False)

In [ ]:
if TRAIN_CRNN:
    if not train_module_exists: raise FileNotFoundError("crnn/train.py not found.")
    if not crnn_config_path.exists(): raise FileNotFoundError(f"Config not found: {crnn_config_path}")
    subprocess.run([sys.executable, "-m", "crnn.train", "--config", CRNN_CONFIG], check=True)
else:
    print("CRNN training skipped. Set TRAIN_CRNN=True when ready.")

## 12. Inspect YOLO dataset configuration

In [ ]:
yolo_config_path = REPO_DIR / "yolo" / "data.yaml"
if yolo_config_path.exists():
    print(yolo_config_path.read_text(errors="replace")[:10000])
else:
    print("yolo/data.yaml not found. Build/prepare the YOLO dataset according to this repository's instructions first.")

## 13. YOLO11 training

Uses Ultralytics and writes runs under `yolo/runs/`. Verify that `yolo/data.yaml` points to the actual dataset before enabling this cell.

In [ ]:
if TRAIN_YOLO:
    if not yolo_config_path.exists(): raise FileNotFoundError("yolo/data.yaml is missing.")
    from ultralytics import YOLO
    model = YOLO(YOLO_MODEL)
    use_device = 0 if ('torch' in globals() and torch.cuda.is_available()) else 'cpu'
    train_results = model.train(
        data=str(yolo_config_path), epochs=YOLO_EPOCHS, imgsz=YOLO_IMAGE_SIZE,
        batch=YOLO_BATCH_SIZE, device=use_device, workers=2,
        project=str(REPO_DIR / "yolo" / "runs"), name="kaggle_starter", exist_ok=True
    )
    print(train_results)
else:
    print("YOLO training skipped. Set TRAIN_YOLO=True when ready.")

## 14. CRNN evaluation

Set the exact checkpoint path. The evaluation script's CLI may vary by revision, so inspect `--help` before running it. This cell assumes the `--weights` argument is supported by the current checkout.

In [ ]:
CRNN_WEIGHTS = None  # e.g. /kaggle/input/crnn-model/crnn_final_model.weights.h5
eval_module_exists = (REPO_DIR / "crnn" / "evaluate.py").exists()
print("crnn/evaluate.py exists:", eval_module_exists)
if eval_module_exists:
    subprocess.run([sys.executable, "-m", "crnn.evaluate", "--help"], check=False)

In [ ]:
if EVALUATE_CRNN:
    if not eval_module_exists: raise FileNotFoundError("crnn/evaluate.py not found.")
    if not CRNN_WEIGHTS: raise ValueError("Set CRNN_WEIGHTS to the exact model checkpoint path first.")
    subprocess.run([sys.executable, "-m", "crnn.evaluate", "--config", CRNN_CONFIG, "--weights", str(CRNN_WEIGHTS)], check=True)
else:
    print("CRNN evaluation skipped. Set EVALUATE_CRNN=True after confirming CLI arguments.")

## 15. YOLO validation

This cell reports detection metrics. For a fair OCR comparison, also use the project's shared evaluation pipeline for CER/WER/sequence accuracy; mAP alone is not equivalent to OCR quality.

In [ ]:
YOLO_WEIGHTS = REPO_DIR / "yolo" / "runs" / "kaggle_starter" / "weights" / "best.pt"
if EVALUATE_YOLO:
    if not yolo_config_path.exists(): raise FileNotFoundError("yolo/data.yaml is missing.")
    if not YOLO_WEIGHTS.exists(): raise FileNotFoundError(f"Weights not found: {YOLO_WEIGHTS}")
    from ultralytics import YOLO
    model = YOLO(str(YOLO_WEIGHTS))
    use_device = 0 if ('torch' in globals() and torch.cuda.is_available()) else 'cpu'
    metrics = model.val(data=str(yolo_config_path), imgsz=YOLO_IMAGE_SIZE, device=use_device)
    print(metrics)
else:
    print("YOLO evaluation skipped. Set EVALUATE_YOLO=True after training or setting YOLO_WEIGHTS.")

## 16. Find checkpoints, reports, logs, and generated data

In [ ]:
output_roots = [
    REPO_DIR / "crnn" / "checkpoints", REPO_DIR / "crnn" / "reports",
    REPO_DIR / "crnn" / "logs", REPO_DIR / "yolo" / "runs",
    REPO_DIR / "synth_pages" / "output"
]
for root in output_roots:
    print(f"\n{root}: {'exists' if root.exists() else 'not found'}")
    if root.exists():
        files = [p for p in root.rglob('*') if p.is_file()]
        for p in files[:40]: print(f"  {p.relative_to(REPO_DIR)} ({p.stat().st_size / (1024**2):.2f} MiB)")
        if len(files) > 40: print(f"  ... and {len(files)-40} more files")

## Troubleshooting

- **`ModuleNotFoundError: No module named 'crnn'`**: change to the repository root before running module commands.
- **GPU missing**: choose a GPU accelerator in Kaggle settings and rerun the runtime check.
- **Wrong dataset paths**: inspect `crnn/training.yaml` and `yolo/data.yaml`; resolve paths relative to the working directory expected by the code.
- **404 when downloading weights**: use the actual output file path or attach the checkpoint as a Kaggle Dataset and use its path under `/kaggle/input/`.
- **Session timeout**: keep checkpoints in `/kaggle/working`, save notebook versions, and avoid regenerating an existing synthetic dataset.
- **Metric comparison**: compare CRNN and YOLO using the same held-out page splits and OCR metrics where possible.